# THE EDGE — Betweenness Centrality (Choice)

**Project:** The Edge, Rambla de Carrasco, Montevideo — Foster + Partners + Ponce de León Architects  
**Owner:** Rania Chihaoui / IAAC  
**Goal:** Compute building-wide **Betweenness Centrality** at `GRID_SIZE = 0.5` and export its heatmap.

Betweenness = how often each space lies on the shortest path between every other pair of spaces in the whole building. **Critical circulation nodes light up.** In The Edge, elevator lobbies and private stair landings are the mandatory bottlenecks for all cross-floor movement — they are expected to score highest. The F1 corridor connecting all 4 shared elevator cores is the predicted high-betweenness zone. Nodes inside single-floor units 102 and 103 will show low betweenness (no through-traffic; all their paths stay on F1).

## Expected runtime — the heaviest of the four
Betweenness is **O(V² × E)**. With ~10 000 nodes at 0.5 m grid (The Edge is boutique-scale, far smaller than Marseille) expect **several hours**. **Leave it running overnight.**

**If it does not finish:** raise `GRID_SIZE` to `1.0` (~2 500 nodes, much faster) and re-run.

**Output:** `assets/TheEdge_spatial_intelligence/06_betweenness_centrality.png`

**How to run:** Run All. Section 11 builds the graph (minutes for The Edge), then the analysis cell does the heavy lifting.

# SETUP — shared boilerplate (run all cells, in order)

The cells in this section are **identical across all four notebooks**. They import the libraries, read `TheEdge_4-Floor-Plans.obj` (4 stacked floor plans at Z = 0, 4, 8, 12), sample the navigable grid at `GRID_SIZE`, and build the **connected building graph** wired through elevator cores and private unit stairs. Run them top-to-bottom before the analysis section below.

> TIP: To smoke-test first, temporarily set `GRID_SIZE = 4.0` in the Configuration cell, confirm everything runs end-to-end, then switch back to 0.5.

## 1. Import the needed libraries

In [ ]:
import os, math, time
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Render every figure as a STATIC image via kaleido (no WebGL). This is what stops
# VS Code's "WebGL is not supported" crash: the interactive webview is never used.
pio.renderers.default = "png"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 3. Configuration

In [ ]:
renderer = "png"
INTERACTIVE_RENDERER = "png"   # set "browser" to rotate 3D figures interactively

BASE_DIR   = r"c:\Users\Win11\GraphML_RaniaChihaoui"
OBJ_PATH   = os.path.join(BASE_DIR, "assets", "obj", "TheEdge_4-Floor-Plans.obj")
ASSETS_DIR = os.path.join(BASE_DIR, "assets", "TheEdge_spatial_intelligence")
os.makedirs(ASSETS_DIR, exist_ok=True)

FLOOR_LEVELS = [0, 4, 8, 12]
FLOOR_NAMES  = ["Ground Floor", "Floor 1", "Floor 2", "Rooftop"]
GRID_SIZE    = 0.5
FLOOR_HEIGHT = 15.0

# ---------------------------------------------------------------------------
# Vertical circulation — The Edge has two types:
#   A) 4 shared elevator cores connecting ALL 4 levels (GF<->F1, F1<->F2, F2<->Rooftop)
#   B) Private internal staircases per duplex / triplex unit
#
# HOW TO GET COORDINATES:
#   In Rhino, place a Point at the centre of each elevator shaft and each stair
#   landing, then read its X, Y from the Properties panel.
#   Replace every (0.0, 0.0) placeholder below with the real values.
# ---------------------------------------------------------------------------

ELEVATOR_CORES = [
    (0.0, 0.0),   # TODO: Core A — measure from Rhino
    (0.0, 0.0),   # TODO: Core B
    (0.0, 0.0),   # TODO: Core C
    (0.0, 0.0),   # TODO: Core D
]
ELEVATOR_LEVEL_PAIRS = [(0, 1), (1, 2), (2, 3)]   # indices into FLOOR_LEVELS

# Private unit stairs: (x, y, floor_index_from, floor_index_to)
# Floor indices: 0=GF, 1=F1, 2=F2, 3=Rooftop
PRIVATE_STAIRS = [
    (0.0, 0.0, 0, 1),   # TODO: Unit 101 — GF <-> F1
    (0.0, 0.0, 0, 1),   # TODO: Unit 104 — GF <-> F1
    (0.0, 0.0, 2, 3),   # TODO: Unit 201 — F2 <-> Rooftop
    (0.0, 0.0, 2, 3),   # TODO: Unit 202 — F2 <-> Rooftop
    (0.0, 0.0, 2, 3),   # TODO: Unit 203 — F2 <-> Rooftop
    (0.0, 0.0, 1, 2),   # TODO: Unit 204 — F1 <-> F2
    (0.0, 0.0, 2, 3),   # TODO: Unit 204 — F2 <-> Rooftop
]

STAIR_LOCATIONS = (
    [(x, y, [(fa, fb)]) for (x, y, fa, fb) in PRIVATE_STAIRS]
    + [(x, y, ELEVATOR_LEVEL_PAIRS) for (x, y) in ELEVATOR_CORES]
)
print(f"{len(ELEVATOR_CORES)} elevator cores × {len(ELEVATOR_LEVEL_PAIRS)} floor pairs "
      f"+ {len(PRIVATE_STAIRS)} private stairs = {len(STAIR_LOCATIONS)} circulation entries")

UNIT_FLOORS = {
    "101": [0, 1], "102": [1], "103": [1], "104": [0, 1],
    "201": [2, 3], "202": [2, 3], "203": [2, 3], "204": [1, 2, 3],
}

SAVE_IMAGES = True

def save_fig(fig, filename):
    if not SAVE_IMAGES or fig is None: return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1800, height=1100, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

* `extract_triangles` / `points_inside` — geometry helpers (pull triangles, point-in-mesh test).
* `find_closest_node` — the instructor's `find_closest_vertex`, snaps a stair location to the nearest grid node.
* `make_cell_face` + `show_face_heatmap` — turn each grid node into a filled square cell and render it as a `Topology.Show` heatmap (S03 export format).

In [ ]:
def extract_triangles(face_list):
    # Return (T,3,2) array of plan-space (X,Y) triangles for a list of topologic faces.
    tris = []
    for f in face_list:
        vs = Topology.Vertices(f)
        pts = [(Vertex.X(v), Vertex.Y(v)) for v in vs]
        for i in range(1, len(pts) - 1):          # fan-triangulate (faces are already triangles)
            tris.append([pts[0], pts[i], pts[i + 1]])
    return np.array(tris)

def points_inside(tris, P):
    # Boolean mask: which points in P (N,2) fall inside ANY triangle of tris (T,3,2).
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    v0 = b - a; v1 = c - a
    d00 = (v0 * v0).sum(1); d01 = (v0 * v1).sum(1); d11 = (v1 * v1).sum(1)
    den = d00 * d11 - d01 * d01
    den[den == 0] = 1e-12
    inside = np.zeros(len(P), bool)
    for i, p in enumerate(P):
        v2 = p - a
        d20 = (v2 * v0).sum(1); d21 = (v2 * v1).sum(1)
        u = (d11 * d20 - d01 * d21) / den
        w = (d00 * d21 - d01 * d20) / den
        if np.any((u >= -1e-6) & (w >= -1e-6) & (u + w <= 1 + 1e-6)):
            inside[i] = True
    return inside

def find_closest_node(node_xy, x, y):
    # Index of the grid node closest to (x, y) -- the instructor's find_closest_vertex.
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, h):
    # A flat square cell of half-size h centred at (cx, cy), z = 0.
    pts = [Vertex.ByCoordinates(cx - h, cy - h, 0.0), Vertex.ByCoordinates(cx + h, cy - h, 0.0),
           Vertex.ByCoordinates(cx + h, cy + h, 0.0), Vertex.ByCoordinates(cx - h, cy + h, 0.0)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

def show_face_heatmap(faces_values, title, filename, colorScale="viridis"):
    # faces_values: list of (face, value). Colours each cell and renders the filled
    # heatmap with Topology.Show, exactly like the S03 notebooks (faceColorKey, black bg).
    vals = [v for _, v in faces_values]
    mn, mx = float(min(vals)), float(max(vals))
    if mx == mn: mx = mn + 1e-9
    for f, val in faces_values:
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Topology.Dictionary(f)
        d = Dictionary.SetValueAtKey(d, "hm_color", col)
        Topology.SetDictionary(f, d)
    faces = [f for f, _ in faces_values]
    fig = Topology.Show(faces, faceColorKey="hm_color", faceOpacity=1.0,
                        showEdges=False, showVertices=False, camera=[0, 0, 6],
                        backgroundColor="black", width=1700, height=1000,
                        showFigure=False, renderer=renderer)
    for fi in range(len(FLOOR_LEVELS)):
        yc = (fi - 1) * ROWGAP
        fig.add_trace(go.Scatter3d(x=[-(UMAX - UMIN) / 2 - 7], y=[yc], z=[0], mode="text",
                                   text=[FLOOR_NAMES[fi]], textfont=dict(color="white", size=16),
                                   showlegend=False))
    fig.update_layout(title=dict(text=title, font=dict(color="white")))
    fig.show(renderer=renderer)
    save_fig(fig, filename)
    return fig

## 5. Import the OBJ and split it into the four floor plans

`Topology.ByOBJPath` returns clusters of triangulated faces. We collect every face and bin it by its centroid's Z value into the four floor levels (Z = 0, 4, 8, 12), then compute the shared plan bounding box.

In [ ]:
result = Topology.ByOBJPath(OBJ_PATH)
all_faces = []
for item in result:
    if Topology.IsInstance(item, "Cluster"):
        cf = Cluster.Faces(item)
        if cf: all_faces.extend(cf)
    elif Topology.IsInstance(item, "Face"):
        all_faces.append(item)
print(f"Imported {len(all_faces)} triangulated faces")

floor_faces = {lv: [] for lv in FLOOR_LEVELS}
for f in all_faces:
    z = Vertex.Z(Topology.Centroid(f))
    lv = min(FLOOR_LEVELS, key=lambda k: abs(k - z))
    floor_faces[lv].append(f)
for i, lv in enumerate(FLOOR_LEVELS):
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_faces[lv])} faces")

# Shared plan bounding box + vertical row gap used to stack floors in the 2D heatmaps
allxy = np.vstack([extract_triangles(floor_faces[lv]).reshape(-1, 2) for lv in FLOOR_LEVELS])
UMIN, VMIN = allxy.min(0)
UMAX, VMAX = allxy.max(0)
UMID, VMID = 0.5 * (UMIN + UMAX), 0.5 * (VMIN + VMAX)
ROWGAP = (VMAX - VMIN) + 8.0
print(f"Plan box: u[{UMIN:.1f},{UMAX:.1f}]  v[{VMIN:.1f},{VMAX:.1f}]")

## 7. Sample a navigable grid on each floor

A regular grid is laid over the shared bounding box; only points inside the meshed (navigable) area are kept. These become the graph nodes of each floor.

In [ ]:
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])

floor_valid = {}    # level -> valid_xy ndarray
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    floor_valid[lv] = GRID_PTS[points_inside(tris, GRID_PTS)]
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_valid[lv])} navigable nodes")

## 9. Build the per-floor graphs, the display cells, and stack everything

For each floor the valid points become topologic vertices at `Z = floor_index * FLOOR_HEIGHT` (for the 3D graph) and a flat square **display cell** centred at `(u, v + offset)` (for the 2D heatmaps). Horizontal edges join 4-neighbour valid points.

In [ ]:
all_v = []            # topologic vertices (all floors, stacked in Z)
all_e = []            # topologic edges
floor_index_map = {}  # level -> {(round u, round v): global vertex index}
display_faces = []    # flat square cells for the heatmaps
cell_lookup = {}      # (floor_index, (round u, round v)) -> display face
H = GRID_SIZE / 2.0

for fi, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    z = fi * FLOOR_HEIGHT
    yoff = (fi - 1) * ROWGAP
    idx = {}
    for (u, v) in valid:
        idx[rk(u, v)] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z)))
        cell = make_cell_face(float(u) - UMID, (float(v) - VMID) + yoff, H)
        display_faces.append(cell)
        cell_lookup[(fi, rk(u, v))] = cell
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f"  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges")
print(f"Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges, {len(display_faces)} display cells")

def heatmap_from_graph(values, title, filename, colorScale):
    # Pair every graph vertex with its display cell, then render the filled heatmap.
    fv = []
    for v, val in zip(gverts, values):
        fi = int(round(Vertex.Z(v) / FLOOR_HEIGHT))
        f = cell_lookup.get((fi, rk(Vertex.X(v), Vertex.Y(v))))
        if f is not None:
            fv.append((f, val))
    return show_face_heatmap(fv, title, filename, colorScale)

## 10. Connect the floors through elevators and private stairs

Two types of vertical edges stitch the four floor plans into one connected building:
- **Elevator cores** (4 shared): each core is connected across all three adjacent floor pairs (GF↔F1, F1↔F2, F2↔Rooftop).
- **Private unit stairs**: unit-specific connections declared in `PRIVATE_STAIRS` — duplexes 101 & 104 (GF↔F1), 201/202/203 (F2↔Rooftop), triplex 204 (F1↔F2 and F2↔Rooftop).

For each circulation point and floor pair we snap to the closest navigable node (`find_closest_node`) and add a vertical edge. Units 102 and 103 (single-floor) are reachable only through the shared elevator core on F1.

In [ ]:
stair_node_pairs = []
for stair in STAIR_LOCATIONS:
    sx, sy = stair[0], stair[1]
    # Use the floor pairs declared for this stair; otherwise connect every adjacent floor.
    pairs = stair[2] if len(stair) >= 3 and stair[2] else [(i, i + 1) for i in range(len(FLOOR_LEVELS) - 1)]
    for fa, fb in pairs:
        a, b = FLOOR_LEVELS[fa], FLOOR_LEVELS[fb]
        va, vb = floor_valid[a], floor_valid[b]
        ia = find_closest_node(va, sx, sy)
        ib = find_closest_node(vb, sx, sy)
        gia = floor_index_map[a][rk(va[ia, 0], va[ia, 1])]
        gib = floor_index_map[b][rk(vb[ib, 0], vb[ib, 1])]
        all_e.append(Edge.ByVertices([all_v[gia], all_v[gib]]))
        stair_node_pairs.append((gia, gib))
print(f"Added {len(stair_node_pairs)} vertical stair edges "
      f"from {len(STAIR_LOCATIONS)} stair location(s)")

## 11. Build the combined BUILDING graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f"Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)")
print(f"Graph density:  {Graph.Density(building_graph):.5f}")

---
# ANALYSIS - this notebook's dedicated workload

## 17. Betweenness Centrality (Choice)

How often each space lies on the shortest paths between all other spaces. High values = critical circulation routes; the stair nodes light up because every cross-floor trip passes through them.

In [ ]:
t0 = time.time()
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True)
a = np.array(betweenness_values, dtype=float)
print(f"Betweenness centrality — {len(betweenness_values)} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} – {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_bc = sorted(zip(gverts, betweenness_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most traversed spaces (critical paths):")
for v, score in sorted_bc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  →  {score:.4f}")

heatmap_from_graph(betweenness_values, "Betweenness Centrality / Choice (whole building)",
                   "06_betweenness_centrality.png", "thermal")